In [1]:
import numpy as np
import torch

from rlaopt.atoms import L1Norm, SumSquares, Affine
from rlaopt.atoms.polyhedron import Polyhedron
from rlaopt.expression import Variable
# from rlaopt.expression.bilevel_expression import BilevelExpression
from rlaopt.solvers.configs import ProxGradConfig
from rlaopt.solvers.proximal_gradient.prox_grad import ProximalGradient
from rlaopt.operator_split import OperatorSplit

In [ ]:
A = torch.randn(10, 100)
C = torch.randn(20, 100)
x_fes = torch.randn(100,)
b = A @ x_fes
l = torch.min(C @ x_fes) * torch.ones(20)
u = torch.max(C @ x_fes) * torch.ones(20)

In [ ]:
x = Variable(torch.ones(100))

In [ ]:
f = Affine(x, A, b)

In [ ]:
type(f.var1)

In [ ]:
f = 0.5 * ((A @ x - b) ** 2).sum()

In [ ]:
P = Polyhedron(x, A=A, b=b, C=C, lower=l, upper=u)

In [ ]:
(A @ x_fes == b).all()

In [ ]:
P.evaluate({"x" :x_fes})

In [17]:
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

In [18]:
# Low-rank matrix completion data generation
Xstar = torch.randn(1000, 10) / (1000 ** 0.5)
Ystar = torch.randn(500, 10) / (500 ** 0.5)
A = Xstar @ Ystar.T

In [19]:
# Initialize variables
X, Y = Variable(torch.randn(1000, 10) / (1000 ** 0.5)), Variable(torch.randn(500, 10) / (500 ** 0.5))

In [20]:
# Setup the objective function
obj = SumSquares(X @ Y.T - A)

In [21]:
# Test objective function evaluation
obj_val = obj.forward()
obj_true = torch.linalg.norm(X.value @ Y.value.T - A, ord='fro') ** 2
print("Error in computed Objective value:", abs(obj_val - obj_true))

Error in computed Objective value: tensor(4.9738e-14, grad_fn=<AbsBackward0>)


In [22]:
# Setup APG with linesearch
config = ProxGradConfig(eta=1.0, tol=1e-6, use_acceleration=True, use_linesearch=True)
opt = ProximalGradient(config, obj)

In [23]:
# Solve with APG step method
params = obj.params
state = opt.init_state(params)
for i in range(200):
    params, state = opt.step(params, state)
    if i % 10 == 0:
        print(f"Iteration {i}, Objective Value: {obj.evaluate(params)}")
print(state.err)

Iteration 0, Objective Value: 13.413015693276039
Iteration 10, Objective Value: 1.9422768506441588
Iteration 20, Objective Value: 0.004236197198553045
Iteration 30, Objective Value: 0.00010528640617145362
Iteration 40, Objective Value: 2.3994600695545825e-06
Iteration 50, Objective Value: 8.084793364853947e-07
Iteration 60, Objective Value: 6.663758379141637e-08
Iteration 70, Objective Value: 1.3330972016066017e-09
Iteration 80, Objective Value: 6.82097523136684e-10
Iteration 90, Objective Value: 1.7305135137355505e-10
Iteration 100, Objective Value: 1.2648250642784545e-11
Iteration 110, Objective Value: 3.9798223722447415e-13
Iteration 120, Objective Value: 3.706143411081748e-13
Iteration 130, Objective Value: 8.126673290194967e-14
Iteration 140, Objective Value: 4.109334777689724e-15
Iteration 150, Objective Value: 3.4278568862195573e-16
Iteration 160, Objective Value: 2.933806773724065e-16
Iteration 170, Objective Value: 4.813263827907348e-17
Iteration 180, Objective Value: 1.617342

In [24]:
names = list(params.keys())

In [25]:
torch.linalg.norm(params[names[0]] @ params[names[1]].T - A, ord='fro') ** 2

tensor(2.5687e-19, grad_fn=<PowBackward0>)

In [26]:
# Lasso data generation
n, p = 1024, 128
ntst = 256
s = 32 
J = np.random.choice(p, s)

xStar = torch.zeros(p)
xStar[J] = torch.randn(s) / (s ** 0.5)
A = torch.randn(n, p) / (n ** 0.5)
Atst = torch.randn(ntst, p) / (ntst ** 0.5)
b = A @ xStar + 0.001 * torch.randn(n) 
btst = Atst @ xStar + 0.001 * torch.randn(ntst)

In [27]:
# Init params + reg
x = Variable(torch.zeros(p))
mu = 0.1 * torch.linalg.norm(A.T @ b, ord=torch.inf)

In [28]:
# Lasso problem
F = SumSquares(A @ x - b) + L1Norm(x, scaling=mu)

In [29]:
f,r = F.operator_split()

In [30]:
F.forward()

tensor(1.0948, grad_fn=<AddBackward0>)

In [31]:
torch.linalg.norm(A @ x.value - b, 2) ** 2 + mu * torch.linalg.norm(x.value, ord=1)

tensor(1.0948, grad_fn=<AddBackward0>)

In [32]:
# Step size is reciprocal of Lipschitz constant
eta = 1 / (2 * torch.linalg.norm(A,ord=2) ** 2)

In [33]:
config = ProxGradConfig(eta=eta, tol=1e-6, use_acceleration=True, use_linesearch=False)
opt = ProximalGradient(config, F)

In [34]:
# Solve the problem using step method
params = F.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

inf
tensor(0.7608, grad_fn=<DivBackward0>)
tensor(0.2696, grad_fn=<DivBackward0>)
tensor(0.0461, grad_fn=<DivBackward0>)
tensor(0.0430, grad_fn=<DivBackward0>)
tensor(0.0338, grad_fn=<DivBackward0>)
tensor(0.0164, grad_fn=<DivBackward0>)
tensor(0.0060, grad_fn=<DivBackward0>)
tensor(0.0036, grad_fn=<DivBackward0>)
tensor(0.0031, grad_fn=<DivBackward0>)
tensor(0.0021, grad_fn=<DivBackward0>)
tensor(0.0010, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0002, grad_fn=<DivBackward0>)
tensor(7.5186e-05, grad_fn=<DivBackward0>)
tensor(6.3815e-05, grad_fn=<DivBackward0>)
tensor(6.6160e-05, grad_fn=<DivBackward0>)
tensor(4.3321e-05, grad_fn=<DivBackward0>)
tensor(1.8122e-05, grad_fn=<DivBackward0>)
tensor(1.1143e-05, grad_fn=<DivBackward0>)
tensor(1.2852e-05, grad_fn=<DivBackward0>)
tensor(9.6792e-06, grad_fn=<DivBackward0>)
tensor(4.7898e-06, grad_fn=<DivBackward0>)
tensor(2.2698e-06, grad_

In [35]:
F.params

{'exprs.0.AddExpression.exprs.0.exprs.1.var4': Parameter containing:
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)}

In [36]:
# Solve the problem using solve method
params, err = opt.solve(F)
# Print norm of gradient mapping
print(err)

tensor(1.1143e-05, grad_fn=<DivBackward0>)


In [37]:
params

{'exprs.0.AddExpression.exprs.0.exprs.1.var4': tensor([-0.0013,  0.0000,  0.0000,  0.1606, -0.0699,  0.3661,  0.0000,  0.0000,
          0.0000,  0.0000,  0.3068,  0.0000,  0.0000, -0.2481,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0554,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000, -0.2051,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0883,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000, -0.1430,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000, -0.1630,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.1845,  0.0000, -0.0194,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.2497,
          0.0000, -0.0090,  0.0000,  0.3989,  0.0000,  0.0000, 

In [38]:
# Expected to be the same as at initialization, since the optimizer does not update in-place
x.value

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [39]:
# Test OperatorSplit interface with the same problem
obj = OperatorSplit(
    SumSquares(A @ x - b),
    L1Norm(x, scaling=mu))

In [40]:
config = ProxGradConfig(eta=eta, tol=1e-6, use_acceleration=True, use_linesearch=False)
opt = ProximalGradient(config, obj)

In [41]:
# Solve the problem using step method
params = obj.f.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

inf
tensor(0.7608, grad_fn=<DivBackward0>)
tensor(0.2696, grad_fn=<DivBackward0>)
tensor(0.0461, grad_fn=<DivBackward0>)
tensor(0.0430, grad_fn=<DivBackward0>)
tensor(0.0338, grad_fn=<DivBackward0>)
tensor(0.0164, grad_fn=<DivBackward0>)
tensor(0.0060, grad_fn=<DivBackward0>)
tensor(0.0036, grad_fn=<DivBackward0>)
tensor(0.0031, grad_fn=<DivBackward0>)
tensor(0.0021, grad_fn=<DivBackward0>)
tensor(0.0010, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0004, grad_fn=<DivBackward0>)
tensor(0.0002, grad_fn=<DivBackward0>)
tensor(7.5186e-05, grad_fn=<DivBackward0>)
tensor(6.3815e-05, grad_fn=<DivBackward0>)
tensor(6.6160e-05, grad_fn=<DivBackward0>)
tensor(4.3321e-05, grad_fn=<DivBackward0>)
tensor(1.8122e-05, grad_fn=<DivBackward0>)
tensor(1.1143e-05, grad_fn=<DivBackward0>)
tensor(1.2852e-05, grad_fn=<DivBackward0>)
tensor(9.6792e-06, grad_fn=<DivBackward0>)
tensor(4.7898e-06, grad_fn=<DivBackward0>)
tensor(2.2698e-06, grad_

In [42]:
# Solve the problem using solve method
params, err = opt.solve(obj)
# Print norm of gradient mapping
print(err)

tensor(1.1143e-05, grad_fn=<DivBackward0>)


In [ ]:
# Test HVP computation
v = torch.randn(p)
Hv = obj.hvp_f(params, v)
print("HVP error:", torch.linalg.norm(Hv - 2 * (A.T @ (A @ v))))

In [ ]:
# Bilevel Expression for optimizing lasso regularization parameter

# Inner function: Lasso on training data
ftr = lambda mu: SumSquares(A @ x - b) + L1Norm(x, mu)

# Outer function: Least squares on test data
ftst = SumSquares(Atst @ x - btst)

# Setup bilevel problem
obj = BilevelExpression(mu, ftr, ftst, config, ProximalGradient)

In [ ]:
# Setup solver for the bilevel problem
config_blvl = ProxGradConfig(
    eta = torch.tensor(0.001), 
    max_iters=500,
    tol=1e-3, 
    use_acceleration=False, 
    use_linesearch=False
)
opt_blvl = ProximalGradient(config_blvl, obj) 

In [ ]:
# Initial objective value
obj.forward()

In [ ]:
# Solve the bilevel problem using solve method
mu_star, err = opt_blvl.solve(obj)

In [ ]:
# Final objective value
obj.evaluate(mu_star)

In [ ]:
# Print the optimal regularization parameter
mu_star

In [ ]:
# Print gradient norm of the objective at mu_star
torch.func.grad(obj.evaluate)(mu_star)["w"].norm().item()